# Notebook 3 — Run 1 Results Analysis

Deep-dive into Run 1 before making any prompt changes.

**Inputs:**
- `outputs/enrichment_results.csv` — per-transaction predictions
- `outputs/enrichment_confusion.csv` — mismatch pair counts
- `outputs/enrichment_accuracy_by_category.csv`
- `outputs/enrichment_accuracy_by_provider.csv`
- `outputs/taxonomy_master.csv` — for spend_type lookups

**Sections:**
1. Setup & data load
2. Headline metrics
3. Confusion matrix — what did the LLM return for 0% categories?
4. True accuracy estimate — manual review sample
5. Spend / non-spend accuracy
6. Invalid key analysis
7. Provider breakdown
8. Ad-hoc query helpers

---
## 1. Setup & Data Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
from pathlib import Path

OUTPUT_DIR = Path("outputs")
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 80)

# ── Load ──────────────────────────────────────────────────────────────────────
df          = pd.read_csv(OUTPUT_DIR / "enrichment_results.csv")
df_conf     = pd.read_csv(OUTPUT_DIR / "enrichment_confusion.csv")
df_cat      = pd.read_csv(OUTPUT_DIR / "enrichment_accuracy_by_category.csv")
df_prov     = pd.read_csv(OUTPUT_DIR / "enrichment_accuracy_by_provider.csv")
df_taxonomy = pd.read_csv(OUTPUT_DIR / "taxonomy_master.csv")

# ── Derived columns ───────────────────────────────────────────────────────────
VALID_KEYS = set(df_taxonomy["base_category_key"].str.strip().str.upper().dropna())

# Spend-type lookup: for mixed keys use 'leans' if available
spend_lookup = (
    df_taxonomy
    .assign(k=lambda x: x["base_category_key"].str.strip().str.upper())
    .assign(st=lambda x: x["leans"].where(x["spend_type"] == "mixed", x["spend_type"]))
    .set_index("k")["st"]
    .to_dict()
)

df["gt_spend"]    = df["ground_truth"].map(spend_lookup)
df["pred_spend"]  = df["llm_prediction"].map(spend_lookup)
df["spend_match"] = np.where(
    df["gt_spend"].notna() & df["pred_spend"].notna() &
    df["gt_spend"].isin(["spend", "non_spend"]) &
    df["pred_spend"].isin(["spend", "non_spend"]),
    df["gt_spend"] == df["pred_spend"],
    np.nan
)

# Convenience subsets — use these throughout for ad-hoc queries
errors   = df[~df["match"]].copy()      # all mismatches
invalids = df[~df["valid_key"]].copy()  # invalid key subset
correct  = df[df["match"]].copy()       # correct predictions

print(f"Loaded {len(df):,} rows  |  {df['provider_name'].nunique()} providers  |  "
      f"{df['ground_truth'].nunique()} GT categories")
print(f"Correct: {df['match'].sum():,} ({df['match'].mean()*100:.1f}%)  |  "
      f"Invalid keys: {(~df['valid_key']).sum():,} ({(~df['valid_key']).mean()*100:.1f}%)")

### a. enrichment results

In [ ]:
df.head().T

### b. enrichment_confusion

In [ ]:
df_conf

### c. enrichment_accuracy_by_category

In [ ]:
df_cat

### d. enrichment_accuracy_by_provider

In [ ]:
df_prov

### e. taxonomy_master

In [ ]:
df_taxonomy

---
## 2. Headline Metrics

In [ ]:
n       = len(df)
n_ok    = df["match"].sum()
n_inv   = (~df["valid_key"]).sum()
n_wrong = n - n_ok - n_inv

assessable       = df["spend_match"].notna() & ~df["match"]
n_spend_ok       = (df.loc[assessable, "spend_match"] == True).sum()

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# ── Left: overall breakdown ───────────────────────────────────────────────────
cats   = ["Exact match", "Wrong key\n(valid)", "Invalid key"]
vals   = [n_ok, n_wrong, n_inv]
colors = ["#2ecc71", "#e67e22", "#e74c3c"]
axes[0].barh(cats, vals, color=colors, height=0.5)
for i, v in enumerate(vals):
    axes[0].text(v + 20, i, f"{v:,}  ({v/n*100:.1f}%)", va="center", fontsize=9)
axes[0].set_xlim(0, n * 1.25)
axes[0].set_title("Overall result breakdown", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Transactions")
axes[0].invert_yaxis()

# ── Middle: accuracy distribution across categories ───────────────────────────
axes[1].hist(df_cat["accuracy"], bins=20, color="#3498db", edgecolor="white", linewidth=0.5)
axes[1].axvline(df_cat["accuracy"].mean(), color="red", linestyle="--",
                linewidth=1.2, label=f"Mean {df_cat['accuracy'].mean():.1f}%")
axes[1].set_title("Accuracy distribution\nacross categories", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Accuracy (%)")
axes[1].set_ylabel("# categories")
axes[1].legend(fontsize=9)

# ── Right: provider accuracy ──────────────────────────────────────────────────
prov_sorted = df_prov.sort_values("accuracy")
bar_colors  = ["#e74c3c" if a < 35 else "#e67e22" if a < 45 else "#2ecc71"
               for a in prov_sorted["accuracy"]]
axes[2].barh(prov_sorted["provider_name"], prov_sorted["accuracy"],
             color=bar_colors, height=0.7)
axes[2].axvline(42.3, color="black", linestyle="--", linewidth=1, label="Overall 42.3%")
axes[2].set_title("Accuracy by provider", fontsize=11, fontweight="bold")
axes[2].set_xlabel("Accuracy (%)")
axes[2].legend(fontsize=9)
axes[2].set_xlim(0, 75)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "nb3_headline.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Of {n - n_ok:,} mismatches, {int(n_spend_ok):,} ({n_spend_ok/(n-n_ok)*100:.1f}%) "
      f"had the correct spend/non-spend type")

**Key observations:**
- Accuracy is bimodally distributed — categories are either very high (>80%) or very low (<20%)
- 0% categories account for a disproportionate share of rows (SAVINGS alone = 188)
- HSBC (25%) and Macquarie (56.5%) are the outlier endpoints — ~30pp spread
- Even on mismatches, the LLM gets spend/non-spend right most of the time (see print above)

---
## 3. Confusion Matrix — 0% Accuracy Categories

For each category with 0% (or near-0%) accuracy, what did the LLM actually return?

Key question: **is the LLM wrong, or is the ground truth wrong?**

In [ ]:
# Categories with accuracy < 15% and n >= 5
focus_cats = (
    df_cat[(df_cat["accuracy"] < 15) & (df_cat["total"] >= 5)]
    .sort_values("total", ascending=False)["ground_truth"]
    .tolist()
)
print(f"Focus categories ({len(focus_cats)}):")
print(focus_cats)

In [ ]:
# ── Heatmap: GT category × LLM prediction ────────────────────────────────────
conf_focus = df_conf[df_conf["ground_truth"].isin(focus_cats)].copy()
top_preds  = conf_focus.groupby("llm_prediction")["count"].sum().nlargest(12).index.tolist()
pivot      = (
    conf_focus[conf_focus["llm_prediction"].isin(top_preds)]
    .pivot_table(index="ground_truth", columns="llm_prediction", values="count", fill_value=0)
)
col_labels = [f"{c} ⚠" if c not in VALID_KEYS else c for c in pivot.columns]

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(col_labels, rotation=40, ha="right", fontsize=8)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=8)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        v = pivot.values[i, j]
        if v > 0:
            ax.text(j, i, str(v), ha="center", va="center",
                    fontsize=7, color="black" if v < pivot.values.max() * 0.6 else "white")
plt.colorbar(im, ax=ax, label="Count")
ax.set_xlabel("LLM Prediction", fontsize=11, fontweight="bold")
ax.set_ylabel("Ground Truth", fontsize=11, fontweight="bold")
ax.set_title("LLM predictions for 0%-accuracy categories  (⚠ = invalid key)",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "nb3_confusion_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
df_conf.ground_truth.unique()

### interogate individual category mismatches

In [ ]:
GT, PRED = "HOME_RENOVATION___MAINTENANCE", "HARDWARE_HOME_IMPROVEMENT"

(
    df.query("ground_truth == @GT and llm_prediction == @PRED")
    [["provider_name", "transaction_type", "account_type", "amount",
      "original_description", "merchant_name", "ground_truth", "llm_prediction"]]
)

In [ ]:
GT, PRED = "SAVINGS", "ROUND_UP"

(
    df.query("ground_truth == @GT and llm_prediction == @PRED")
    [["provider_name", "transaction_type", "account_type", "amount",
      "original_description", "merchant_name", "ground_truth", "llm_prediction"]]
)

In [ ]:
# ── Per-category printed detail ───────────────────────────────────────────────
print("=" * 70)
for cat in focus_cats:
    row      = df_cat[df_cat["ground_truth"] == cat].iloc[0]
    cat_conf = df_conf[df_conf["ground_truth"] == cat].sort_values("count", ascending=False)
    print(f"\n{cat}  (n={row['total']:.0f}, accuracy={row['accuracy']:.1f}%)")
    for _, r in cat_conf.iterrows():
        inv = "  ⚠ INVALID" if r["llm_prediction"] not in VALID_KEYS else ""
        print(f"  → {r['llm_prediction']:<40} {int(r['count']):>4}{inv}")
print("=" * 70)

In [ ]:
# ── Drill-in: inspect actual transactions for any GT → prediction pair ────────
# Change GT and PRED to any values you want to inspect
GT, PRED = "SAVINGS", "ROUND_UP"

(
    df.query("ground_truth == @GT and llm_prediction == @PRED")
    [["provider_name", "transaction_type", "account_type", "amount",
      "original_description", "merchant_name", "ground_truth", "llm_prediction"]]
    .head(20)
)

---
## 4. True Accuracy Estimate — Manual Review Sample

Stratified sample of mismatches for human review. Fill in `reviewer_verdict`:
- `LLM_CORRECT` — LLM is right, ground truth is wrong
- `GT_CORRECT` — Ground truth is right, LLM is wrong
- `BOTH_OK` — Genuinely ambiguous, both defensible
- `NEITHER` — Both are wrong

In [ ]:
# Label each mismatch with an error category for stratified sampling
CONDITIONS = [
    (errors["ground_truth"] == "SAVINGS",                                         "savings_gt_issue"),
    (errors["ground_truth"] == "UNCATEGORISED",                                   "uncategorised_gt"),
    (~errors["valid_key"],                                                         "invalid_key"),
    (errors["ground_truth"].isin(["BAR", "CAFE", "TAKEAWAY___SNACKS"]) &
     errors["llm_prediction"].isin(["RESTAURANT", "GROCERIES"]),                  "near_miss_food"),
    (errors["ground_truth"].isin(["INTERNAL_TRANSFER", "EXTERNAL_TRANSFER_IN",
                                   "EXTERNAL_TRANSFER_OUT"]),                      "transfer_confusion"),
]
errors["error_type"] = "other_mismatch"
for mask, label in reversed(CONDITIONS):
    errors.loc[mask, "error_type"] = label

# Stratified sample: proportional, ~40 rows
SAMPLE_N = 40
parts = [
    grp.sample(min(max(1, round(SAMPLE_N * len(grp) / len(errors))), len(grp)), random_state=42)
    for _, grp in errors.groupby("error_type")
]
review = pd.concat(parts).drop_duplicates().head(SAMPLE_N)

REVIEW_COLS = [
    "transaction_id", "provider_name", "transaction_type", "account_type",
    "amount", "original_description", "merchant_name",
    "ground_truth", "llm_prediction", "valid_key", "error_type",
]
review_out = review[REVIEW_COLS].copy()
review_out["reviewer_verdict"] = ""  # fill in manually
review_out["notes"]            = ""

review_out.to_csv(OUTPUT_DIR / "manual_review_sample.csv", index=False)
print(f"Saved {len(review_out)} rows → outputs/manual_review_sample.csv")
print()
print("Error type breakdown:")
print(review_out["error_type"].value_counts().to_string())

# ── Load completed review & calculate true accuracy estimate ──────────────────
# Run after filling in manual_review_sample.csv

REVIEW_FILE = OUTPUT_DIR / "manual_review_sample.csv"
reviewed    = pd.read_csv(REVIEW_FILE)
filled      = reviewed[reviewed["reviewer_verdict"].str.strip().ne("")]

if len(filled) == 0:
    print("No verdicts yet — fill in manual_review_sample.csv first.")
else:
    vc      = filled["reviewer_verdict"].value_counts()
    llm_ok  = vc.get("LLM_CORRECT", 0)
    gt_ok   = vc.get("GT_CORRECT",  0)
    both_ok = vc.get("BOTH_OK",     0)
    neither = vc.get("NEITHER",     0)
    n_rev   = len(filled)

    base_correct, base_total = 1354, 3200
    extra   = (base_total - base_correct) * llm_ok  / n_rev
    partial = (base_total - base_correct) * both_ok / n_rev * 0.5

    print(f"Reviewed: {n_rev}  |  LLM_CORRECT={llm_ok}  GT_CORRECT={gt_ok}  "
          f"BOTH_OK={both_ok}  NEITHER={neither}")
    print()
    print(f"Reported accuracy   : {base_correct/base_total*100:.1f}%")
    print(f"Conservative est.   : {(base_correct+extra)/base_total*100:.1f}%  "
          f"(counting LLM_CORRECT mismatches as correct)")
    print(f"Generous est.       : {(base_correct+extra+partial)/base_total*100:.1f}%  "
          f"(half-credit for BOTH_OK)")
    print()
    print("Verdict by error_type:")
    print(filled.groupby(["error_type", "reviewer_verdict"]).size().unstack(fill_value=0).to_string())

---
## 5. Spend / Non-Spend Accuracy

Did the LLM at least get spend vs non-spend right, even when the exact key was wrong?
- If yes → precision problem (taxonomy granularity), prompt tuning should help
- If no → comprehension problem, more fundamental rethink needed

In [ ]:
assessable       = df["spend_match"].notna()
mismatch_assess  = ~df["match"] & assessable

overall_spend_acc  = df.loc[assessable,         "spend_match"].mean()
mismatch_spend_acc = df.loc[mismatch_assess,    "spend_match"].mean()

n_exact    = df["match"].sum()
n_sp_ok    = (df.loc[mismatch_assess, "spend_match"] == True).sum()
n_sp_wrong = (df.loc[mismatch_assess, "spend_match"] == False).sum()
n_inv_keys = (~df["valid_key"]).sum()
n_unkn     = (~assessable & ~df["match"]).sum()

print(f"Spend/non-spend accuracy (all assessable rows) : {overall_spend_acc*100:.1f}%")
print(f"Spend/non-spend accuracy (mismatches only)     : {mismatch_spend_acc*100:.1f}%")
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Left: stacked breakdown of all 3,200 rows ─────────────────────────────────
labels = ["Exact key\ncorrect", "Wrong key,\ncorrect spend", "Wrong key,\nwrong spend",
          "Invalid key", "Unknown\nspend type"]
sizes  = [n_exact, int(n_sp_ok), int(n_sp_wrong), n_inv_keys, int(n_unkn)]
colors = ["#2ecc71", "#a8d8a8", "#e74c3c", "#c0392b", "#bdc3c7"]
axes[0].bar(labels, sizes, color=colors)
for i, s in enumerate(sizes):
    axes[0].text(i, s + 15, f"{s}\n({s/3200*100:.1f}%)", ha="center", fontsize=8)
axes[0].set_title("Breakdown of all 3,200 transactions", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Count")
axes[0].set_ylim(0, max(sizes) * 1.2)

# ── Right: spend acc vs exact acc for worst 15 categories ────────────────────
sp_by_cat = (
    df[assessable]
    .groupby("ground_truth")
    .agg(n=("spend_match", "count"), spend_ok=("spend_match", "sum"), exact_ok=("match", "sum"))
    .query("n >= 5")
    .assign(spend_acc=lambda x: x["spend_ok"] / x["n"] * 100,
            exact_acc=lambda x: x["exact_ok"] / x["n"] * 100)
    .sort_values("spend_acc")
    .head(15)
    .reset_index()
)
y = range(len(sp_by_cat))
axes[1].barh(y, sp_by_cat["spend_acc"],  color="#3498db", height=0.4, label="Spend/non-spend acc")
axes[1].barh([i + 0.4 for i in y], sp_by_cat["exact_acc"], color="#e67e22", height=0.4, label="Exact key acc")
axes[1].set_yticks([i + 0.2 for i in y])
axes[1].set_yticklabels(sp_by_cat["ground_truth"], fontsize=8)
axes[1].set_xlabel("Accuracy (%)")
axes[1].set_title("Spend acc vs exact acc\n(worst 15 categories)", fontsize=11, fontweight="bold")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "nb3_spend_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Inspect spend-type failures: GT=spend but LLM predicted non_spend ─────────
# These are the most concerning — LLM misunderstood the transaction type entirely
(
    df.query("spend_match == False and gt_spend == 'spend'")
    [["provider_name", "original_description", "merchant_name",
      "ground_truth", "gt_spend", "llm_prediction", "pred_spend"]]
    .head(20)
)

---
## 6. Invalid Key Analysis

270 transactions (8.4%) where the LLM returned a key not in the taxonomy.
- Are these near-misses (truncated valid keys) or hallucinations?
- Which ground truth categories and providers generate the most?

In [ ]:
inv_counts = (
    invalids["llm_prediction"]
    .value_counts()
    .rename_axis("invalid_key")
    .reset_index(name="n")
    .assign(pct=lambda x: (x["n"] / len(invalids) * 100).round(1))
)

CLASSIFY = {
    "SPEND": "top_level_only", "NON_SPEND": "top_level_only",
    "TRANSPORT": "truncated",  "TRANSFER": "truncated",
    "INSURANCE": "truncated",  "FEES": "truncated",
    "ENTERTAINMENT": "hallucination", "SHOPPING_RETAIL": "hallucination",
    "HARDWARE_HOME_IMPROVEMENT": "hallucination",
    "NOTES": "provider_specific",
}
inv_counts["type"] = inv_counts["invalid_key"].map(CLASSIFY).fillna("other")

print(f"Total invalid: {len(invalids):,}  |  Distinct invalid keys: {inv_counts['invalid_key'].nunique()}")
print()
print(inv_counts.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: top invalid keys coloured by classification ────────────────────────
type_colors = {
    "top_level_only": "#e74c3c", "truncated": "#e67e22",
    "hallucination":  "#9b59b6", "provider_specific": "#3498db", "other": "#95a5a6"
}
top_inv  = inv_counts.head(15)
bar_cols = [type_colors[t] for t in top_inv["type"]]
axes[0].barh(top_inv["invalid_key"], top_inv["n"], color=bar_cols)
axes[0].set_xlabel("Count")
axes[0].set_title("Top invalid keys", fontsize=11, fontweight="bold")
axes[0].invert_yaxis()
legend_els = [Patch(facecolor=c, label=l) for l, c in type_colors.items()]
axes[0].legend(handles=legend_els, fontsize=8, loc="lower right")

# ── Right: invalid key rate by GT category ────────────────────────────────────
inv_by_gt = (
    invalids.groupby("ground_truth").size()
    .reset_index(name="n_invalid")
    .merge(df_cat[["ground_truth", "total"]], on="ground_truth")
    .assign(inv_pct=lambda x: x["n_invalid"] / x["total"] * 100)
    .query("total >= 5")
    .sort_values("n_invalid", ascending=False)
    .head(15)
)
axes[1].barh(inv_by_gt["ground_truth"], inv_by_gt["inv_pct"], color="#9b59b6")
for i, (_, r) in enumerate(inv_by_gt.iterrows()):
    axes[1].text(r["inv_pct"] + 0.3, i, f"{int(r['n_invalid'])}", va="center", fontsize=8)
axes[1].set_xlabel("Invalid key rate (%)")
axes[1].set_title("Invalid key rate by GT category\n(n per bar = count)",
                  fontsize=11, fontweight="bold")
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "nb3_invalid_keys.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Drill into a specific invalid key ─────────────────────────────────────────
# Change INVALID_KEY to any key from the table above
INVALID_KEY = "HARDWARE_HOME_IMPROVEMENT"

(
    invalids.query("llm_prediction == @INVALID_KEY")
    [["provider_name", "original_description", "merchant_name",
      "ground_truth", "llm_prediction"]]
    .head(20)
)

---
## 7. Provider Breakdown

- Which providers are structural outliers vs caught up in category-level noise?
- For worst providers: what specific errors are driving low accuracy?

In [ ]:
prov_detail = (
    df.groupby("provider_name")
    .agg(
        total   =("match",     "count"),
        correct =("match",     "sum"),
        invalid =("valid_key", lambda x: (~x).sum()),
    )
    .assign(
        accuracy    =lambda x: (x["correct"] / x["total"] * 100).round(1),
        invalid_pct =lambda x: (x["invalid"] / x["total"] * 100).round(1),
        wrong_valid =lambda x: x["total"] - x["correct"] - x["invalid"],
    )
    .sort_values("accuracy")
    .reset_index()
)

# ── Stacked bar: correct / wrong-valid / invalid per provider ─────────────────
fig, ax = plt.subplots(figsize=(11, 5))
x = range(len(prov_detail))
ax.bar(x, prov_detail["correct"],     label="Correct",           color="#2ecc71")
ax.bar(x, prov_detail["wrong_valid"], label="Wrong (valid key)", color="#e67e22",
       bottom=prov_detail["correct"])
ax.bar(x, prov_detail["invalid"],     label="Invalid key",       color="#e74c3c",
       bottom=prov_detail["correct"] + prov_detail["wrong_valid"])
ax.set_xticks(x)
ax.set_xticklabels(prov_detail["provider_name"], rotation=35, ha="right", fontsize=9)
ax.axhline(200, color="black", linewidth=0.8, linestyle="--", label="n=200 per provider")
ax.set_ylabel("Transactions")
ax.set_title("Per-provider result breakdown  (sorted by accuracy)",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "nb3_provider_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()

print(prov_detail[["provider_name", "total", "correct", "accuracy",
                    "wrong_valid", "invalid", "invalid_pct"]].to_string(index=False))

In [ ]:
# ── Top mismatches for any provider ───────────────────────────────────────────
# Change PROVIDER to any provider name
PROVIDER = "HSBC"

(
    df.query("provider_name == @PROVIDER and not match")
    .groupby(["ground_truth", "llm_prediction"])
    .size()
    .reset_index(name="count")
    .assign(valid=lambda x: x["llm_prediction"].isin(VALID_KEYS))
    .sort_values("count", ascending=False)
    .head(15)
)

In [ ]:
# ── HSBC deep-dive: GT distribution vs other providers ────────────────────────
# Does HSBC have a skewed category distribution that explains low accuracy?
hsbc_gt = df[df["provider_name"] == "HSBC"]["ground_truth"].value_counts().rename("HSBC")
all_gt  = df[df["provider_name"] != "HSBC"]["ground_truth"].value_counts().div(15).rename("Others (avg)")

pd.concat([hsbc_gt, all_gt.round(0)], axis=1).fillna(0).sort_values("HSBC", ascending=False).head(20)

---
## 8. Ad-Hoc Query Helpers

Change the parameter at the top of each cell and re-run.

In [ ]:
# ── A) All mismatches for a specific GT category ──────────────────────────────
GT_CAT = "INTERNAL_TRANSFER"

(
    errors.query("ground_truth == @GT_CAT")
    [["provider_name", "transaction_type", "account_type", "amount",
      "original_description", "merchant_name", "llm_prediction", "valid_key"]]
    .sort_values("llm_prediction")
    .head(30)
)

In [ ]:
# ── B) What GT categories does a specific LLM key get pulled from? ────────────
LLM_KEY = "GROCERIES"

(
    errors.query("llm_prediction == @LLM_KEY")
    .groupby("ground_truth")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

In [ ]:
# ── C) Sample transactions where spend type was wrong ────────────────────────
(
    df.query("spend_match == False")
    [["provider_name", "original_description", "merchant_name",
      "ground_truth", "gt_spend", "llm_prediction", "pred_spend"]]
    .sample(20, random_state=1)
)

In [ ]:
# ── D) Free-text search across descriptions ───────────────────────────────────
SEARCH = "bunnings"

(
    df[df["original_description"].str.lower().str.contains(SEARCH, na=False)]
    [["provider_name", "original_description", "merchant_name",
      "ground_truth", "llm_prediction", "match", "valid_key"]]
)

In [ ]:
# ── E) Provider × prediction pivot for a specific GT category ────────────────
GT_CAT = "SAVINGS"

(
    df.query("ground_truth == @GT_CAT")
    .groupby(["provider_name", "llm_prediction"])
    .size()
    .unstack(fill_value=0)
    .assign(TOTAL=lambda x: x.sum(axis=1))
    .sort_values("TOTAL", ascending=False)
    .drop(columns="TOTAL")
)

In [ ]:
# ── F) Export a clean error table for sharing / annotation ───────────────────
# Set EXPORT_CAT to None to export all errors
EXPORT_CAT = "SAVINGS"

export_df = errors if EXPORT_CAT is None else errors.query("ground_truth == @EXPORT_CAT")
out_path  = OUTPUT_DIR / f"errors_{'all' if EXPORT_CAT is None else EXPORT_CAT}.csv"
export_df[
    ["transaction_id", "provider_name", "transaction_type", "account_type", "amount",
     "original_description", "merchant_name", "ground_truth", "llm_prediction", "valid_key"]
].to_csv(out_path, index=False)
print(f"Exported {len(export_df):,} rows → {out_path}")

### Interactive Error Analysis

In [ ]:
!pip install anywidget

In [ ]:
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)

In [ ]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
import pandas as pd

# ── Build pivot ───────────────────────────────────────────────────────────────
conf_focus = df_conf[df_conf["ground_truth"].isin(focus_cats)].copy()
top_preds  = conf_focus.groupby("llm_prediction")["count"].sum().nlargest(20).index.tolist()
pivot      = (
    conf_focus[conf_focus["llm_prediction"].isin(top_preds)]
    .pivot_table(index="ground_truth", columns="llm_prediction", values="count", fill_value=0)
)
pivot_pct  = pivot.div(pivot.sum(axis=1), axis=0) * 100
col_labels = [f"{c} ⚠" if c not in VALID_KEYS else c for c in pivot.columns]
row_labels = [
    f"{cat} (n={int(df_cat.loc[df_cat['ground_truth']==cat, 'total'].values[0])})"
    for cat in pivot.index
]

# ── Labels store ──────────────────────────────────────────────────────────────
labels_store  = {}   # (gt, pred, txn_id) → {verdict, notes}
combo_store   = {}   # (gt, pred)         → {verdict, notes}

# ── Heatmap figure ────────────────────────────────────────────────────────────
def make_heatmap(use_pct):
    data       = pivot_pct.values if use_pct else pivot.values
    text_vals  = [[f"{v:.0f}%" if use_pct else str(int(v)) if v > 0 else ""
                   for v in row] for row in data]
    cbar_label = "% of GT category" if use_pct else "Count"
    return go.Heatmap(
        z=data,
        x=col_labels,
        y=row_labels,
        colorscale="YlOrRd",
        text=text_vals,
        texttemplate="%{text}",
        hovertemplate="GT: %{y}<br>Pred: %{x}<br>" +
                      ("%{z:.1f}%" if use_pct else "%{z:.0f}") + "<extra></extra>",
        showscale=True,
        colorbar=dict(title=cbar_label),
    )

fig = go.FigureWidget(make_heatmap(use_pct=True))
fig.update_layout(
    title="Click a cell to review transactions",
    xaxis=dict(title="LLM Prediction", tickangle=90, tickfont=dict(size=9)),
    yaxis=dict(title="Ground Truth",   tickfont=dict(size=9)),
    height=700,
    width=1100,
    margin=dict(l=220, b=180, t=50),
)

# ── Toggle ────────────────────────────────────────────────────────────────────
toggle = widgets.ToggleButtons(
    options=["% of GT category", "Count"],
    description="Show:",
    button_style="",
    layout=widgets.Layout(margin="8px 0"),
)

def on_toggle(change):
    use_pct = change["new"] == "% of GT category"
    with fig.batch_update():
        new_trace = make_heatmap(use_pct)
        fig.data[0].z             = new_trace.z
        fig.data[0].text          = new_trace.text
        fig.data[0].texttemplate  = new_trace.texttemplate
        fig.data[0].hovertemplate = new_trace.hovertemplate
        fig.data[0].colorbar      = new_trace.colorbar

toggle.observe(on_toggle, names="value")

# ── Review panel ──────────────────────────────────────────────────────────────
selection_label = widgets.HTML("<b>Click a cell in the heatmap to load transactions</b>")
progress_html   = widgets.HTML("")

# ── Combo-level verdict ───────────────────────────────────────────────────────
combo_header  = widgets.HTML("<b>Combo-level label</b> (applies to all transactions in this cell)")
combo_verdict = widgets.Dropdown(
    options=["", "GT_ERROR", "LLM_ERROR", "TAXONOMY_GAP", "AMBIGUOUS", "MIXED"],
    description="Verdict:",
    layout=widgets.Layout(width="300px"),
)
combo_notes  = widgets.Textarea(
    placeholder="Notes about this GT → prediction pair overall...",
    description="Notes:",
    layout=widgets.Layout(width="95%", height="55px"),
)
combo_save_btn    = widgets.Button(description="Save combo label", button_style="warning", icon="tag")
combo_save_status = widgets.HTML("")

# ── Transaction-level verdict ─────────────────────────────────────────────────
txn_header   = widgets.HTML("<b>Transaction-level label</b>")
txn_selector = widgets.Dropdown(description="Transaction:", layout=widgets.Layout(width="95%"))
txn_detail   = widgets.HTML("")
txn_verdict  = widgets.Dropdown(
    options=["", "LLM_CORRECT", "GT_CORRECT", "BOTH_OK", "NEITHER"],
    description="Verdict:",
    layout=widgets.Layout(width="300px"),
)
txn_notes    = widgets.Textarea(
    placeholder="Optional notes for this transaction...",
    description="Notes:",
    layout=widgets.Layout(width="95%", height="55px"),
)
txn_save_btn    = widgets.Button(description="Save txn label", button_style="success", icon="check")
txn_save_status = widgets.HTML("")

# ── Export ────────────────────────────────────────────────────────────────────
export_btn    = widgets.Button(description="Export all labels", button_style="primary", icon="download")
export_status = widgets.HTML("")

# ── State ─────────────────────────────────────────────────────────────────────
current_gt   = [None]
current_pred = [None]
current_txns = [None]

# ── Helpers ───────────────────────────────────────────────────────────────────
def fmt_detail(row):
    fields = {
        "Provider":       row.get("provider_name", ""),
        "Txn type":       row.get("transaction_type", ""),
        "Account type":   row.get("account_type", ""),
        "Amount":         row.get("amount", ""),
        "Description":    row.get("original_description", ""),
        "Merchant":       row.get("merchant_name", ""),
        "Ground truth":   f"<b style='color:#e74c3c'>{row.get('ground_truth','')}</b>",
        "LLM prediction": f"<b style='color:#2980b9'>{row.get('llm_prediction','')}</b>",
    }
    rows = "".join(
        f"<tr><td style='padding:3px 10px;color:#666;white-space:nowrap'><b>{k}</b></td>"
        f"<td style='padding:3px'>{v}</td></tr>"
        for k, v in fields.items()
    )
    return f"<table style='font-size:13px;border-collapse:collapse'>{rows}</table>"

def update_progress():
    gt, pred = current_gt[0], current_pred[0]
    txns = current_txns[0]
    if txns is None or len(txns) == 0:
        return
    labelled = sum(1 for tid in txns["transaction_id"] if (gt, pred, tid) in labels_store)
    combo_done = "🏷️ combo labelled" if (gt, pred) in combo_store else ""
    progress_html.value = (
        f"Txn labels: <b>{labelled} / {len(txns)}</b> &nbsp;&nbsp; {combo_done}"
    )

def refresh_txn_dropdown():
    gt, pred = current_gt[0], current_pred[0]
    txns = current_txns[0]
    if txns is None:
        return
    cur = txn_selector.value
    txn_selector.options = [
        (f"[{'✅' if (gt, pred, row['transaction_id']) in labels_store else '  '}]  "
         f"{row['transaction_id']}  |  {str(row.get('original_description',''))[:60]}",
         i)
        for i, (_, row) in enumerate(txns.iterrows())
    ]
    if cur is not None and cur < len(txns):
        txn_selector.value = cur

def load_cell(gt_label, pred_label):
    gt_clean   = gt_label.split(" (n=")[0]
    pred_clean = pred_label.rstrip(" ⚠")
    current_gt[0]   = gt_clean
    current_pred[0] = pred_clean

    txns = df[
        (df["ground_truth"]   == gt_clean) &
        (df["llm_prediction"] == pred_clean)
    ].reset_index(drop=True)
    current_txns[0] = txns

    selection_label.value = (
        f"<b style='font-size:14px'>"
        f"GT: <span style='color:#e74c3c'>{gt_clean}</span> &nbsp;→&nbsp; "
        f"Pred: <span style='color:#2980b9'>{pred_clean}</span>"
        f" &nbsp; ({len(txns)} transactions)</b>"
    )

    # Load existing combo verdict
    existing_combo = combo_store.get((gt_clean, pred_clean), {})
    combo_verdict.value = existing_combo.get("verdict", "")
    combo_notes.value   = existing_combo.get("notes",   "")
    combo_save_status.value = ""

    if len(txns) == 0:
        txn_selector.options = []
        txn_detail.value     = ""
        update_progress()
        return

    txn_selector.options = [
        (f"[{'✅' if (gt_clean, pred_clean, row['transaction_id']) in labels_store else '  '}]  "
         f"{row['transaction_id']}  |  {str(row.get('original_description',''))[:60]}",
         i)
        for i, (_, row) in enumerate(txns.iterrows())
    ]
    txn_selector.value = 0
    update_progress()

# ── Event handlers ────────────────────────────────────────────────────────────
def on_click(trace, points, state):
    if not points.point_inds:
        return
    load_cell(points.ys[0], points.xs[0])

fig.data[0].on_click(on_click)

def on_txn_select(change):
    idx = change["new"]
    if idx is None or current_txns[0] is None:
        return
    row = current_txns[0].iloc[idx]
    txn_detail.value = fmt_detail(row)
    key = (current_gt[0], current_pred[0], row["transaction_id"])
    existing = labels_store.get(key, {})
    txn_verdict.value  = existing.get("verdict", "")
    txn_notes.value    = existing.get("notes",   "")
    txn_save_status.value = ""

txn_selector.observe(on_txn_select, names="value")

def on_save_combo(b):
    gt, pred = current_gt[0], current_pred[0]
    if gt is None:
        return
    combo_store[(gt, pred)] = {"verdict": combo_verdict.value, "notes": combo_notes.value}
    combo_save_status.value = "✅ Saved"
    update_progress()

combo_save_btn.on_click(on_save_combo)

def on_save_txn(b):
    idx = txn_selector.value
    if idx is None or current_txns[0] is None:
        return
    row = current_txns[0].iloc[idx]
    key = (current_gt[0], current_pred[0], row["transaction_id"])
    labels_store[key] = {"verdict": txn_verdict.value, "notes": txn_notes.value}
    txn_save_status.value = "✅ Saved"
    refresh_txn_dropdown()
    update_progress()

txn_save_btn.on_click(on_save_txn)

def on_export(b):
    out_rows = []

    # Combo-level labels
    combo_rows = [
        {"type": "combo", "ground_truth": gt, "llm_prediction": pred,
         "transaction_id": "", "verdict": v["verdict"], "notes": v["notes"]}
        for (gt, pred), v in combo_store.items()
    ]

    # Transaction-level labels
    txn_rows = [
        {"type": "transaction", "ground_truth": gt, "llm_prediction": pred,
         "transaction_id": tid, "verdict": v["verdict"], "notes": v["notes"]}
        for (gt, pred, tid), v in labels_store.items()
    ]

    out = pd.DataFrame(combo_rows + txn_rows)
    if len(out) == 0:
        export_status.value = "⚠️ No labels saved yet"
        return
    out.to_csv(OUTPUT_DIR / "manual_review_labels.csv", index=False)
    export_status.value = (
        f"✅ Exported {len(combo_rows)} combo + {len(txn_rows)} txn labels "
        f"→ outputs/manual_review_labels.csv"
    )

export_btn.on_click(on_export)

# ── Layout ────────────────────────────────────────────────────────────────────
combo_panel = widgets.VBox([
    combo_header,
    widgets.HBox([combo_verdict, combo_save_btn, combo_save_status]),
    combo_notes,
], layout=widgets.Layout(padding="8px", border="1px solid #f39c12",
                          border_radius="4px", margin="6px 0"))

txn_panel = widgets.VBox([
    txn_header,
    txn_selector,
    txn_detail,
    widgets.HBox([txn_verdict, txn_save_btn, txn_save_status]),
    txn_notes,
], layout=widgets.Layout(padding="8px", border="1px solid #27ae60",
                          border_radius="4px", margin="6px 0"))

review_panel = widgets.VBox([
    selection_label,
    progress_html,
    combo_panel,
    txn_panel,
    widgets.HBox([export_btn, export_status]),
], layout=widgets.Layout(padding="10px", width="100%"))

display(widgets.VBox([toggle, fig, review_panel]))
